In [43]:
import os
print(os.getcwd())

/Users/irammukri/Desktop/pde4444_project/notebooks


In [44]:
import os
import zipfile
import shutil
from PIL import Image
from pillow_heif import register_heif_opener

register_heif_opener()

RAW_DIR = "../dataset/raw"
classes = ["PASS", "FAIL"]

# -----------------------------
# 1. EXTRACT ZIP FILES
# -----------------------------
for cls in classes:
    cls_path = os.path.join(RAW_DIR, cls)

    for file in os.listdir(cls_path):
        if file.endswith(".zip"):
            zip_path = os.path.join(cls_path, file)
            print(f"Extracting {zip_path}...")

            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(cls_path)

            # Delete zip after extraction
            os.remove(zip_path)
            print(f"Deleted zip: {file}")

# -----------------------------
# 2. FIX NESTED FOLDERS
# -----------------------------
for cls in classes:
    cls_path = os.path.join(RAW_DIR, cls)

    for item in os.listdir(cls_path):
        item_path = os.path.join(cls_path, item)

        if os.path.isdir(item_path):
            print(f"Fixing nested folder: {item}")

            for file in os.listdir(item_path):
                shutil.move(os.path.join(item_path, file), cls_path)

            os.rmdir(item_path)

# -----------------------------
# 3. CONVERT HEIC → JPG
# -----------------------------
for cls in classes:
    cls_path = os.path.join(RAW_DIR, cls)

    for file in os.listdir(cls_path):
        if file.lower().endswith(".heic"):
            heic_path = os.path.join(cls_path, file)
            jpg_path = os.path.splitext(heic_path)[0] + ".jpg"

            try:
                image = Image.open(heic_path).convert("RGB")
                image.save(jpg_path, "JPEG", quality=95)

                os.remove(heic_path)
                print(f"Converted: {file} → JPG")

            except Exception as e:
                print(f"Failed: {file} → {e}")

# -----------------------------
# 4. FINAL VERIFICATION
# -----------------------------
print("\n📊 FINAL DATASET CHECK")

for cls in classes:
    cls_path = os.path.join(RAW_DIR, cls)
    files = os.listdir(cls_path)

    print(f"{cls}: {len(files)} images")

    # Show first few files
    print("Sample:", files[:5])
    print("-" * 30)

Fixing nested folder: .ipynb_checkpoints
Converted: IMG_1482.HEIC → JPG
Converted: IMG_1553.HEIC → JPG
Converted: IMG_1504.HEIC → JPG
Converted: IMG_1565.HEIC → JPG
Converted: IMG_1660.HEIC → JPG
Converted: IMG_1677.HEIC → JPG
Converted: IMG_1548.HEIC → JPG
Converted: IMG_1572.HEIC → JPG
Converted: IMG_1552.HEIC → JPG
Converted: IMG_1694.HEIC → JPG
Converted: IMG_1513.HEIC → JPG
Converted: IMG_1704.HEIC → JPG
Converted: IMG_1616.HEIC → JPG
Converted: IMG_1483.HEIC → JPG
Converted: IMG_1600.HEIC → JPG
Converted: IMG_1657.HEIC → JPG
Converted: IMG_1467.HEIC → JPG
Converted: IMG_1559.HEIC → JPG
Converted: IMG_1596.HEIC → JPG
Converted: IMG_1580.HEIC → JPG
Converted: IMG_1685.HEIC → JPG
Converted: IMG_1543.HEIC → JPG
Converted: IMG_1503.HEIC → JPG
Converted: IMG_1554.HEIC → JPG
Converted: IMG_1606.HEIC → JPG
Converted: IMG_1651.HEIC → JPG
Converted: IMG_1671.HEIC → JPG
Converted: IMG_1667.HEIC → JPG
Converted: IMG_1688.HEIC → JPG
Converted: IMG_1519.HEIC → JPG
Converted: IMG_1466.HEIC → JP

In [45]:
import os
import shutil

src = "../dataset/raw/PASS_aug"
dst = "../dataset/raw/PASS"

for file in os.listdir(src):
    src_file = os.path.join(src, file)
    dst_file = os.path.join(dst, file)
    
    # If file already exists, rename it to avoid overwrite
    if os.path.exists(dst_file):
        base, ext = os.path.splitext(file)
        dst_file = os.path.join(dst, base + "_aug" + ext)
    
    shutil.move(src_file, dst_file)

print("Done merging PASS_aug into PASS ✅")

Done merging PASS_aug into PASS ✅


In [46]:
import os

print("PASS before:", len(os.listdir("../dataset/raw/PASS")))
print("FAIL before:", len(os.listdir("../dataset/raw/FAIL")))

PASS before: 461
FAIL before: 537


In [10]:
import os
import cv2
import numpy as np
from tqdm import tqdm

In [11]:
RAW_DIR = "../dataset/raw"
PROCESSED_DIR = "dataset/processed"

classes = ["PASS", "FAIL"]
IMG_SIZE = 224

In [12]:
for cls in classes:
    os.makedirs(os.path.join(PROCESSED_DIR, cls), exist_ok=True)

In [13]:
data = []
labels = []

for cls in classes:
    path = os.path.join(RAW_DIR, cls)
    label = 1 if cls == "PASS" else 0

    for img_name in tqdm(os.listdir(path)):
        img_path = os.path.join(path, img_name)

        img = cv2.imread(img_path)

        if img is None:
            continue

        # Resize
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        # Normalize
        img = img / 255.0

        save_path = os.path.join(PROCESSED_DIR, cls, img_name)
        cv2.imwrite(save_path, (img * 255).astype(np.uint8))

        data.append(img)
        labels.append(label)

100%|█████████████████████████████████████████| 537/537 [00:13<00:00, 38.58it/s]


In [14]:
import os

print(os.listdir("../dataset/raw"))

['.DS_Store', 'FAIL', 'PASS', 'PASS_aug', '.ipynb_checkpoints']


In [15]:
X = np.array(data)
y = np.array(labels)

print(X.shape, y.shape)

(996, 224, 224, 3) (996,)


In [16]:
from sklearn.model_selection import train_test_split

# First split: Train + Temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# Second split: Validation + Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

Train: (697, 224, 224, 3)
Val: (149, 224, 224, 3)
Test: (150, 224, 224, 3)


In [17]:
np.save("../dataset/X_train.npy", X_train)
np.save("../dataset/y_train.npy", y_train)

np.save("../dataset/X_val.npy", X_val)
np.save("../dataset/y_val.npy", y_val)

np.save("../dataset/X_test.npy", X_test)
np.save("../dataset/y_test.npy", y_test)

In [55]:
import os

os.makedirs("processed_dataset/PASS", exist_ok=True)
os.makedirs("processed_dataset/FAIL", exist_ok=True)

print("Folders created")

Folders created


In [56]:
import os
print(os.listdir("../dataset/raw"))

['.DS_Store', 'FAIL', 'PASS', 'PASS_aug', '.ipynb_checkpoints']


In [57]:
import shutil

shutil.rmtree("processed_dataset")
print("Old processed_dataset deleted")

Old processed_dataset deleted


In [61]:
import cv2
import numpy as np
import os

os.makedirs("processed_dataset/PASS", exist_ok=True)
os.makedirs("processed_dataset/FAIL", exist_ok=True)

def save_images(X, y, prefix):
    for i in range(len(X)):
        img = X[i]

        # tensor → numpy
        if hasattr(img, "permute"):
            img = img.permute(1, 2, 0).cpu().numpy()

        elif isinstance(img, np.ndarray):
            if img.shape[0] == 3:
                img = np.transpose(img, (1, 2, 0))

        # normalize
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        img = (img * 255).astype("uint8")

        label = y[i].item() if hasattr(y[i], "item") else int(y[i])

        if label == 1:
            path = f"processed_dataset/PASS/{prefix}_pass_{i}.jpg"
        else:
            path = f"processed_dataset/FAIL/{prefix}_fail_{i}.jpg"

        cv2.imwrite(path, img)

#  save ALL splits
save_images(X_train, y_train, "train")
save_images(X_val, y_val, "val")
save_images(X_test, y_test, "test")

print("ALL processed images saved!")

ALL processed images saved!


In [18]:
import os

pass_count = len(os.listdir("../dataset/raw/PASS"))
fail_count = len(os.listdir("../dataset/raw/FAIL"))

print("Processed Dataset:")
print("PASS images:", pass_count)
print("FAIL images:", fail_count)
print("Total images:", pass_count + fail_count)

Raw Dataset:
PASS images: 461
FAIL images: 537
Total images: 998


In [20]:
X = np.array(data)
y = np.array(labels)

In [21]:
# CNN input
print("\nCNN Input Shape:", X.shape[1:])

# Traditional ML (flattened)
flattened_dim = X.shape[1] * X.shape[2] * X.shape[3]
print("Traditional ML Feature Dimension (N):", flattened_dim)


CNN Input Shape: (224, 224, 3)
Traditional ML Feature Dimension (N): 150528
